# 5-Wege-Benchmark: Decoder-Only vs. Encoder-Decoder Modelle

Dieses Notebook visualisiert die Ergebnisse des 5-Wege-Modellvergleichs auf dem Lebenshilfe-Testset:
1. **Decoder-Only Basismodell (`Qwen2.5-1.5B`):** Unberührte **Few-Shot Baseline** mit In-Context-Learning.
2. **Decoder-Only SFT Modell:** Supervised Fine-Tuned LoRA Adapter auf Qwen.
3. **Decoder-Only DPO Modell:** Preference-Aligned LoRA Adapter auf Qwen.
4. **Encoder-Decoder SFT Modell:** Feinabgestimmtes `mBART-50`.
5. **Encoder-Decoder DPO Modell:** Direct Preference Optimized `mBART-50`.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

def find_repo_root():
    p = os.path.abspath(os.getcwd())
    while p != os.path.dirname(p):
        if os.path.exists(os.path.join(p, 'data')) and os.path.exists(os.path.join(p, 'results')):
            return p
        p = os.path.dirname(p)
    return os.path.abspath(os.path.expanduser('~/Documents/Master Thesis'))

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print('Arbeitsverzeichnis:', os.getcwd())


## 1. 5-Wege-Benchmark-Ergebnisse laden

In [ ]:
CSV_PATH = os.path.join(REPO_ROOT, 'results/evaluation/benchmark_5way_decoder_vs_encoder_decoder.csv')
if not os.path.exists(CSV_PATH):
    CSV_PATH = 'results/evaluation/benchmark_5way_decoder_vs_encoder_decoder.csv'

df_eval = pd.read_csv(CSV_PATH)
print(f'5-Wege-Benchmark erfolgreich geladen: {len(df_eval)} Artikel aus {CSV_PATH}')
display(df_eval.head())


## 2. Wissenschaftliche 5-Wege-Gesamtvergleichstabelle
Systematischer Vergleich über alle 5 Modellierungsansätze.

In [ ]:
summary_csv = os.path.join(REPO_ROOT, 'results/evaluation/master_benchmark_summary.csv')

if os.path.exists(summary_csv):
    df_master_table = pd.read_csv(summary_csv)
    print('=' * 105)
    print('  WISSENSCHAFTLICHE GESAMT-VERGLEICHSTABELLE (5-WEGE-BENCHMARK, N=37)')
    print('=' * 105)
    display(df_master_table)
else:
    def format_mean_std(values):
        return f'{np.mean(values):.4f} ± {np.std(values):.4f}'

    models_config = [
        ('1. Few-Shot Baseline (Qwen-1.5B)', 'r_style_dec_fs', 'r_sem_as_dec_fs', 'composite_dec_fs', 'lix_dec_fs', 'flesch_dec_fs'),
        ('2. Decoder-Only SFT (Qwen-1.5B)', 'r_style_dec_sft', 'r_sem_as_dec_sft', 'composite_dec_sft', 'lix_dec_sft', 'flesch_dec_sft'),
        ('3. Decoder-Only DPO (Qwen-1.5B)', 'r_style_dec_dpo', 'r_sem_as_dec_dpo', 'composite_dec_dpo', 'lix_dec_dpo', 'flesch_dec_dpo'),
        ('4. Encoder-Decoder SFT (mBART-50)', 'r_style_enc_sft', 'r_sem_as_enc_sft', 'composite_enc_sft', 'lix_enc_sft', 'flesch_enc_sft'),
        ('5. Encoder-Decoder DPO (mBART-50)', 'r_style_enc_dpo', 'r_sem_as_enc_dpo', 'composite_enc_dpo', 'lix_enc_dpo', 'flesch_enc_dpo'),
    ]

    rows = []
    for name, c_st, c_sm, c_tot, c_lix, c_fl in models_config:
        rows.append({
            'Modell / Paradigma': name,
            'Simplicity (R_style)': format_mean_std(df_eval[c_st]),
            'SBERT-Quelltreue (R_sem)': format_mean_std(df_eval[c_sm]),
            'Composite Total Reward': format_mean_std(df_eval[c_tot]),
            'LIX Index (↓)': format_mean_std(df_eval[c_lix]),
            'Flesch DE (↑)': format_mean_std(df_eval[c_fl]),
        })

    df_master_table = pd.DataFrame(rows)
    print('=' * 105)
    print('  WISSENSCHAFTLICHE GESAMT-VERGLEICHSTABELLE (5-WEGE-BENCHMARK, N=37)')
    print('=' * 105)
    display(df_master_table)


## 3. Publikationsreife Visualisierungen (Boxplots über alle 5 Modellvarianten)

In [ ]:
sns.set_theme(style="whitegrid", font_scale=1.05)
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

plot_data = []
short_names = [
    ("Dec-FewShot", "r_style_dec_fs", "r_sem_as_dec_fs", "composite_dec_fs"),
    ("Dec-SFT", "r_style_dec_sft", "r_sem_as_dec_sft", "composite_dec_sft"),
    ("Dec-DPO", "r_style_dec_dpo", "r_sem_as_dec_dpo", "composite_dec_dpo"),
    ("EncDec-SFT", "r_style_enc_sft", "r_sem_as_enc_sft", "composite_enc_sft"),
    ("EncDec-DPO", "r_style_enc_dpo", "r_sem_as_enc_dpo", "composite_enc_dpo"),
]

for model_name, col_style, col_sem, col_tot in short_names:
    for _, row in df_eval.iterrows():
        plot_data.append({
            "Modell": model_name,
            "Simplicity (R_style)": row[col_style],
            "SBERT-Treue (R_sem)": row[col_sem],
            "Composite Reward": row[col_tot],
        })

df_plot = pd.DataFrame(plot_data)
palette = ["#3498db", "#2ecc71", "#9b59b6", "#e67e22", "#e74c3c"]

# Simplicity
sns.boxplot(ax=axes[0], data=df_plot, x="Modell", y="Simplicity (R_style)", palette=palette)
axes[0].set_title("A) Simplicity Score (BiLSTM)", fontweight="bold")
axes[0].set_ylim(0, 1.05)
axes[0].tick_params(axis='x', rotation=25)

sns.boxplot(ax=axes[1], data=df_plot, x="Modell", y="SBERT-Treue (R_sem)", palette=palette)
axes[1].set_title("B) Semantische Quelltreue (Jina SBERT)", fontweight="bold")
axes[1].set_ylim(0, 1.05)
axes[1].tick_params(axis='x', rotation=25)

# Composite Reward
sns.boxplot(ax=axes[2], data=df_plot, x="Modell", y="Composite Reward", palette=palette)
axes[2].set_title("C) Composite Total Reward", fontweight="bold")
axes[2].set_ylim(0, 1.05)
axes[2].tick_params(axis='x', rotation=25)

plt.tight_layout()
plot_path = "results/plots/benchmark_5way_decoder_vs_encoder_decoder.png"
plt.savefig(plot_path, dpi=300)
print(f"Vergleichs-Plot gespeichert unter: {plot_path}")
plt.show()

## 4. Qualitatives Side-by-Side Stichproben-Audit (Beispielartikel)

In [ ]:
sep = '=' * 105
print(sep)
print('  QUALITATIVE STICHPROBEN: SIDE-BY-SIDE VERGLEICH ALLER 5 MODELLE')
print(sep)

def clean_snippet(text, max_len=300):
    t = str(text or '').strip()
    if t.endswith('...'):
        t = t[:-3].strip()
    if len(t) > max_len:
        return t[:max_len].strip() + ' [...]'
    return t

as_col = 'source_text' if 'source_text' in df_eval.columns else 'as_text'
ref_col = 'target_text' if 'target_text' in df_eval.columns else 'ls_ref_text'
t_fs_col = 'gen_dec_fs' if 'gen_dec_fs' in df_eval.columns else 'translation_dec_fs'
t_sft_dec_col = 'gen_dec_sft' if 'gen_dec_sft' in df_eval.columns else 'translation_dec_sft'
t_dpo_dec_col = 'gen_dec_dpo' if 'gen_dec_dpo' in df_eval.columns else 'translation_dec_dpo'
t_sft_enc_col = 'gen_enc_sft' if 'gen_enc_sft' in df_eval.columns else 'translation_enc_sft'
t_dpo_enc_col = 'gen_enc_dpo' if 'gen_enc_dpo' in df_eval.columns else 'translation_enc_dpo'

for idx in range(min(5, len(df_eval))):
    row = df_eval.iloc[idx]
    title = row.get('title', row.get('id', f'Artikel {idx+1}'))

    print('')
    print(sep)
    print(f'[ARTIKEL {idx+1}]: {title}')
    print(sep)
    print('[1. AS-QUELLE]:')
    print(clean_snippet(row[as_col]))
    print('')
    print('[2. LEBENSHILFE GOLD REFERENZ]:')
    print(clean_snippet(row[ref_col]))
    print('')
    print(f'[3. FEW-SHOT BASELINE (Qwen-1.5B)] (Style: {row["r_style_dec_fs"]:.3f} | Sem: {row["r_sem_as_dec_fs"]:.3f} | Comp: {row["composite_dec_fs"]:.3f}):')
    print(clean_snippet(row[t_fs_col]))
    print('')
    print(f'[4. DECODER-ONLY SFT (Qwen-1.5B)] (Style: {row["r_style_dec_sft"]:.3f} | Sem: {row["r_sem_as_dec_sft"]:.3f} | Comp: {row["composite_dec_sft"]:.3f}):')
    print(clean_snippet(row[t_sft_dec_col]))
    print('')
    print(f'[5. DECODER-ONLY DPO (Qwen-1.5B)] (Style: {row["r_style_dec_dpo"]:.3f} | Sem: {row["r_sem_as_dec_dpo"]:.3f} | Comp: {row["composite_dec_dpo"]:.3f}):')
    print(clean_snippet(row[t_dpo_dec_col]))
    print('')
    print(f'[6. ENCODER-DECODER SFT (mBART-50)] (Style: {row["r_style_enc_sft"]:.3f} | Sem: {row["r_sem_as_enc_sft"]:.3f} | Comp: {row["composite_enc_sft"]:.3f}):')
    print(clean_snippet(row[t_sft_enc_col]))
    print('')
    print(f'[7. ENCODER-DECODER DPO (mBART-50)] (Style: {row["r_style_enc_dpo"]:.3f} | Sem: {row["r_sem_as_enc_dpo"]:.3f} | Comp: {row["composite_enc_dpo"]:.3f}):')
    print(clean_snippet(row[t_dpo_enc_col]))
    print('-' * 105)
